# Pre-processing Raw ADNI Dataset
- Convert .nii format MRI images to png

In [12]:
#import libraries
import os
import nibabel as nib
import numpy as np
from PIL import Image
import shutil

In [5]:
# Define the input and output directories
input_dir = "../ADNI"  # Folder containing .nii or .nii.gz files
output_dir = "../2D_AXIAL"  # Folder where slices will be saved
classes = ["AD", "CN", "MCI"] # Categories

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Loop through all NIfTI files in the input directory
for cat in classes:
    input_dir_cat = input_dir + "/" + cat
    output_dir_cat = output_dir + "/" + cat
    os.makedirs(output_dir_cat, exist_ok=True)
    for nii_file in os.listdir(input_dir_cat):
        if nii_file.endswith(".nii") or nii_file.endswith(".nii.gz"):  # Process only NIfTI files
            nii_path = os.path.join(input_dir_cat, nii_file)

            try:
                # Load the NIfTI image
                nii_img = nib.load(nii_path)
                img_data = nii_img.get_fdata()
        
                # Normalize intensity values for better contrast (optional)
                img_data = (img_data - img_data.min()) / (img_data.max() - img_data.min()) * 255
                img_data = img_data.astype(np.uint8)
        
                # Create a subfolder for this file's slices
                subfolder = os.path.join(output_dir_cat, os.path.splitext(nii_file)[0])
                os.makedirs(subfolder, exist_ok=True)
        
                # Save axial slices
                for i in range(img_data.shape[2]):  # Loop through axial slices (along the 3rd axis)
                    slice_img = img_data[:, :, i]  # Extract the ith axial slice
                    slice_path = os.path.join(subfolder, f"slice_{i:03d}.png")
                    Image.fromarray(slice_img).save(slice_path)
        
                print(f"Processed {nii_file} and saved slices in {subfolder}")
                
            except Exception as e:
                print(f"Error processing {nii_file}: {e}. Skipping this file.")

print("All files have been processed!")

Processed I118924.nii and saved slices in ../2D_AXIAL/AD/I118924
Processed I64862.nii and saved slices in ../2D_AXIAL/AD/I64862
Processed I79577.nii and saved slices in ../2D_AXIAL/AD/I79577
Processed I89653.nii and saved slices in ../2D_AXIAL/AD/I89653
Processed I65374.nii and saved slices in ../2D_AXIAL/AD/I65374
Processed I122703.nii and saved slices in ../2D_AXIAL/AD/I122703
Processed I97069.nii and saved slices in ../2D_AXIAL/AD/I97069
Processed I38887.nii and saved slices in ../2D_AXIAL/AD/I38887
Processed I72805.nii and saved slices in ../2D_AXIAL/AD/I72805
Processed I106507.nii and saved slices in ../2D_AXIAL/AD/I106507
Processed I31326.nii and saved slices in ../2D_AXIAL/AD/I31326
Processed I83806.nii and saved slices in ../2D_AXIAL/AD/I83806
Processed I59677.nii and saved slices in ../2D_AXIAL/AD/I59677
Processed I118881.nii and saved slices in ../2D_AXIAL/AD/I118881
Processed I118880.nii and saved slices in ../2D_AXIAL/AD/I118880
Processed I82700.nii and saved slices in ../2

In [7]:
for cat in classes:
    output_dir_cat = output_dir + "/" + cat
    num_subfolders = 0
    total_files = 0

    # Iterate through the main folder
    for entry in os.scandir(output_dir_cat):
        if entry.is_dir():  # Check if it's a subfolder
            num_subfolders += 1
            # Count files in this subfolder
            total_files += sum(1 for _ in os.scandir(entry.path) if _.is_file())

    print(f"Number of {cat} subfolders: {num_subfolders}")
    print(f"Total number of files across all {cat} subfolders: {total_files}")

Number of AD subfolders: 80
Total number of files across all AD subfolders: 13154
Number of CN subfolders: 136
Total number of files across all CN subfolders: 22430
Number of MCI subfolders: 35
Total number of files across all MCI subfolders: 5742


# Sorting ADNI Data into categories

In [117]:
import os
import shutil

# Define paths
dataset_root = "../ADNI"  # Change this to your dataset path
output_root = "../ADNI_Sorted"  # Where sorted data will be stored
category_files = {
    "AD": "../subject_lists/AD.txt",
    "CN": "../subject_lists/CN.txt",
    "MCI": "../subject_lists/MCI.txt"
}

# Create output directories if they don't exist
for category in category_files.keys():
    os.makedirs(os.path.join(output_root, category), exist_ok=True)

# Read subject lists
subject_dict = {}
for category, file_path in category_files.items():
    with open(file_path, "r") as f:
        subject_dict[category] = set(line.strip() for line in f.readlines())

# Traverse the dataset
for root, dirs, files in os.walk(dataset_root):
    for subject_id in dirs:  # subject_id subfolders found
        # Now that we're at the subject folder level, check if it's in the category lists
        for category, subjects in subject_dict.items():
            if subject_id in subjects:
                # Form destination path
                dest_path = os.path.join(output_root, category, subject_id)
                
                # If the subject folder exists, move it to the appropriate category folder
                src_path = os.path.join(root, subject_id)
                if os.path.isdir(src_path):
                    shutil.move(src_path, dest_path)
                break  # No need to check other categories once moved

print("Sorting complete!")

Sorting complete!


In [118]:
import os
import shutil

# Define paths
dataset_root = "../ADNI_Sorted"  # Change this to your dataset path
output_root = "../ADNI_Cleaned"  # New location for cleaned dataset

# Ensure output root exists
os.makedirs(output_root, exist_ok=True)

# Traverse AD, MCI, CN categories
for category in os.listdir(dataset_root):
    category_path = os.path.join(dataset_root, category)
    
    if not os.path.isdir(category_path):
        continue

    # Create the corresponding category folder in the output directory
    category_output_path = os.path.join(output_root, category)
    os.makedirs(category_output_path, exist_ok=True)

    # Loop through each subject in the category
    for subject in os.listdir(category_path):
        subject_path = os.path.join(category_path, subject)
        
        if not os.path.isdir(subject_path):
            continue

        # print(f"Processing subject: {subject} in {category}")

        # Search deeper for nii files inside id # folders
        for processing_folder in os.listdir(subject_path):
            processing_path = os.path.join(subject_path, processing_folder)
            if not os.path.isdir(processing_path):
                continue

            for date_folder in os.listdir(processing_path):
                date_path = os.path.join(processing_path, date_folder)
                if not os.path.isdir(date_path):
                    continue

                for id_folder in os.listdir(date_path):
                    id_path = os.path.join(date_path, id_folder)
                    if not os.path.isdir(id_path):
                        continue

                    # Create an output directory using the id #
                    id_output_path = os.path.join(category_output_path, id_folder)
                    os.makedirs(id_output_path, exist_ok=True)

                    # Move and rename .nii files
                    for file in os.listdir(id_path):
                        if file.endswith(".nii"):
                            src_file = os.path.join(id_path, file)
                            
                            # Extract everything after the last underscore
                            new_filename = file.split("_")[-1]  # Get last part
                            new_file_path = os.path.join(id_output_path, new_filename)

                            shutil.move(src_file, new_file_path)
                            # print(f"Renamed and moved: {file} -> {new_filename}")

        # Optionally, remove empty subject folders
        shutil.rmtree(subject_path, ignore_errors=True)

print("Dataset restructuring complete!")

Dataset restructuring complete!


# Pre-processing using GAN architecture method

In [ ]:
import os
import shutil
import numpy as np  # Ensure numpy is imported

# Define the input and output directories
input_dir = "../ADNI_Cleaned"  # Folder containing subject folders with .nii files
split_dir = "../ttv_split_ADNI"  # Folder where split files will be saved
classes = ["AD", "CN", "MCI"]  # Categories

# Create train and test directories
train_dir = os.path.join(split_dir, "train")
test_dir = os.path.join(split_dir, "test")

for cat in classes:
    input_dir_cat = os.path.join(input_dir, cat)  # Path to category folder
    train_cat_dir = os.path.join(train_dir, cat)
    test_cat_dir = os.path.join(test_dir, cat)

    # Create category subfolders for train/test
    os.makedirs(train_cat_dir, exist_ok=True)
    os.makedirs(test_cat_dir, exist_ok=True)

    # Loop through each subject folder inside the category folder
    for subject in os.listdir(input_dir_cat):
        subject_path = os.path.join(input_dir_cat, subject)

        if not os.path.isdir(subject_path):  # Ensure it's a folder
            continue

        split = np.random.rand()  # Generate random number for splitting

        if split <= 0.80:  # 80% to train
            subject_dest = os.path.join(train_cat_dir, subject)
        else:  # 20% to test
            subject_dest = os.path.join(test_cat_dir, subject)

        # Create the subject directory inside train/test
        os.makedirs(subject_dest, exist_ok=True)

        # Move all .nii files to the corresponding split directory
        for nii_file in os.listdir(subject_path):
            if nii_file.endswith(".nii") or nii_file.endswith(".nii.gz"):  # Ensure it's a NIfTI file
                src = os.path.join(subject_path, nii_file)
                dst = os.path.join(subject_dest, nii_file)

                shutil.move(src, dst)  # Move instead of copy
                # print(f"Moved {nii_file} to {subject_dest}")

        # Remove empty subject folder after moving all files
        if not os.listdir(subject_path):  # Check if folder is empty
            os.rmdir(subject_path)
            # print(f"Removed empty folder: {subject_path}")

print("All files have been processed!")

Moved I87150.nii to ../ttv_split_ADNI/train/AD/I87150
Removed empty folder: ../ADNI_Cleaned/AD/I87150
Moved I85469.nii to ../ttv_split_ADNI/train/AD/I85469
Removed empty folder: ../ADNI_Cleaned/AD/I85469
Moved I47177.nii to ../ttv_split_ADNI/train/AD/I47177
Removed empty folder: ../ADNI_Cleaned/AD/I47177
Moved I64827.nii to ../ttv_split_ADNI/train/AD/I64827
Removed empty folder: ../ADNI_Cleaned/AD/I64827
Moved I34195.nii to ../ttv_split_ADNI/test/AD/I34195
Removed empty folder: ../ADNI_Cleaned/AD/I34195
Moved I138797.nii to ../ttv_split_ADNI/train/AD/I138797
Removed empty folder: ../ADNI_Cleaned/AD/I138797
Moved I94432.nii to ../ttv_split_ADNI/train/AD/I94432
Removed empty folder: ../ADNI_Cleaned/AD/I94432
Moved I79577.nii to ../ttv_split_ADNI/train/AD/I79577
Removed empty folder: ../ADNI_Cleaned/AD/I79577
Moved I77044.nii to ../ttv_split_ADNI/train/AD/I77044
Removed empty folder: ../ADNI_Cleaned/AD/I77044
Moved I86377.nii to ../ttv_split_ADNI/train/AD/I86377
Removed empty folder: ../A

# Convert nii images to png

In [123]:
import nibabel as nib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import PIL.Image as Image
import pandas as pd
import os
from os import walk, path, makedirs

In [124]:
# Specify whether you're processing "ad" (Alzheimer's) or "nor" (Normal) category
# dtype = "AD"  # Change to "nor" when processing normal subjects
img_dir = "../ttv_split_ADNI_png/"  # Output directory for PNG images
nifti_dir = "../ttv_split_ADNI"   # Input directory containing .nii files (train/test)

# A sample filename structure in ADNI
# Example: ttv_split_ad/test/ad/005_S_0814/MPR__GradWarp__B1_Correction__N3__Scaled/2006-08-30_09_32_32.0/S18390/...
#          .../ADNI_005_S_0814_MR_MPR__GradWarp__B1_Correction__N3__Scaled_Br_20070923123111793_S18390_I74591.nii"

In [125]:
def create_directory(filename):
    """
    Create the directory structure in the output folder based on input NIfTI file paths.
    Returns:
        for_csv: The path for CSV recording.
        current_dir: The directory where the .png files will be saved.
    """
    dir_list = filename.split("/")

    name = dir_list[-1].split('.')[0]  # Extract the base filename (excluding .nii)
    dir_list = dir_list[2:-1]  # Remove the last part of the path to avoid repeating the filename

    # Set the base output directory
    current_dir = img_dir
    for next_dir in dir_list:
        current_dir = os.path.join(current_dir, next_dir)
        if not os.path.isdir(current_dir):
            os.makedirs(current_dir)  # Create the directory if it does not exist

    # Return both CSV path and the final output directory where PNG images will be stored
    return current_dir, os.path.join(current_dir, name)

In [126]:
def make_images(destination, data):
    """
    Convert the 3D volumetric data into multiple 2D axial slices and save them as PNG files.
    """
    images = []
    
    # Extract axial slices (2D slices along the first axis)
    for i in range(data.shape[0]):
        array = np.array(data[i, :, :])  # Get a single axial slice
        images.append(array)

    # Normalize the images
    norm_images = []
    for array in images:
        max_element = np.amax(array)
        if max_element > 0:
            array = (array / max_element) * 255.0  # Normalize to 0-255 range
        norm_images.append(array)

    # Save images as PNG
    for img, i in zip(norm_images, range(len(norm_images))):
        im = Image.fromarray(img)
        if im.mode != 'L':
            im = im.convert('L')  # Convert to grayscale (if not already)
        im.save(f"{destination}_{len(norm_images) - 1 - i}.png")  # Save with reversed order (bottom to top)

In [127]:
# List to store file paths and dimensions for the CSV file
name_list = []
count_list = []

# Walk through the directory and process each NIfTI file
for root, _, files in walk(nifti_dir):
    for file in files:
        if file.endswith('.nii'):
            filename = os.path.join(root, file)
            
            try:
                img = nib.load(filename)  # Load NIfTI image
            except Exception as e:
                print(f"Error loading NIfTI file: {e}")
            
            data = img.get_fdata()  # Convert to numpy array (use get_fdata instead of get_data)

            # Create directories and generate images
            for_csv, destination = create_directory(filename)
            make_images(destination, data)

            # Record the directory and shape of the data
            name_list.append(for_csv)
            count_list.append(data.shape)

# Create a DataFrame to save the paths and dimensions into a CSV
df = pd.DataFrame(data={"dir_name": name_list, "file_count": count_list})
df.to_csv(f"./counter_all_ttv_{dtype}.csv", sep=',', index=False)

print("All files have been processed and converted to PNG!")

All files have been processed and converted to PNG!


# Select relevant sequences of MRI images only (relevant to AD)

In [ ]:
# The effects of AD are visible mostly on the middle part of the head, with the lower and upper portions being irrelevant.
# With that in mind, we empirically chose some thresholds and only kept the relevant middle parts (from the ~45 percentile slice to the ~80 percentile slice).
# However, the original volumetric data, and thus the .png data we used, did not have the same dimensions
# i.e. the subjects' heads were not split in the same number of slices, and said slices did not have the same dimensions.
# So, if the visit had 192 slices, only slices 92-140 were kept, etc.

import shutil
from os import walk, path, makedirs

# Define the directories for the dataset
png_dir = "../ttv_split_ADNI_png"  # Root directory containing the train/test/AD/MCI/CN structure
seq_dir = "../ttv_split_ADNI_png_seq"  # Directory where the relevant slices will be copied

# Loop through the files in the png_dir
for root, _, files in walk(png_dir):
    # Check if the directory is a category directory (AD, MCI, CN)
    if "/AD/" in root:
        dtype = "AD"
    elif "/CN/" in root:
        dtype = "CN"
    elif "/MCI/" in root:
        dtype = "MCI"
    else:
        continue  # Skip any directories that don't belong to AD, MCI, CN

    # Check if it's a "train" or "test" directory
    if "test" in root:
        split = "test"
    elif "train" in root:
        split = "train"
    else:
        continue  # Skip if it's neither train nor test

    # Ensure the split category directory exists (train/test -> AD/CN/MCI)
    current_dir = path.join(seq_dir, split, dtype)
    if not path.isdir(current_dir):
        makedirs(current_dir, exist_ok=True)

    # Process each file in the current directory
    for file in files:
        if file.endswith(".png"):  # Only process .png files
            # Extract the subject ID and sequence (slice number) from the filename
            file_lst = file.split("_")
            subject_id = file_lst[0]  # Subject ID is the part before the first underscore
            sequence = file_lst[-1].split('.')[0]  # Slice number part (after the last underscore)

            # Determine the number of slices (files) for this subject
            num_slices = len(files)

            # Apply thresholds for the slices based on the total number of slices
            # Modify the threshold ranges based on your dataset and slice count
            if num_slices == 192:
                # Keep slices from 95 to 140 for 192 slices
                if 95 <= int(sequence) <= 140:
                    dst_dir = path.join(seq_dir, split, dtype, subject_id)  # Path for the subject ID folder
                    if not path.isdir(dst_dir):
                        makedirs(dst_dir, exist_ok=True)  # Ensure the ID folder exists
                    shutil.move(path.join(root, file), path.join(dst_dir, file))  # Copy the file

            elif num_slices == 240:
                # Keep slices from 100 to 180 for 240 slices
                if 100 <= int(sequence) <= 180:
                    dst_dir = path.join(seq_dir, split, dtype, subject_id)
                    if not path.isdir(dst_dir):
                        makedirs(dst_dir, exist_ok=True)
                    shutil.move(path.join(root, file), path.join(dst_dir, file))  # Copy the file

            elif num_slices == 256:
                # Keep slices from 120 to 200 for 256 slices
                if 120 <= int(sequence) <= 200:
                    dst_dir = path.join(seq_dir, split, dtype, subject_id)
                    if not path.isdir(dst_dir):
                        makedirs(dst_dir, exist_ok=True)
                    shutil.move(path.join(root, file), path.join(dst_dir, file))  # Copy the file

print("Relevant slices have been moved!")

Relevant slices have been copied!


# Resize images and save

In [130]:
import numpy as np
from os import path, makedirs, walk
from PIL import Image

In [131]:
def center_crop_resize(image_path, output_path, target_size=128):
    image = Image.open(image_path)

    # Get current size
    width, height = image.size

    # Find the smaller dimension to crop a square
    crop_size = min(width, height)

    # Calculate crop box (center crop)
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size

    # Crop and resize
    image = image.crop((left, top, right, bottom))
    image = image.resize((target_size, target_size), Image.LANCZOS)

    # Save the resized image
    image.save(output_path)

In [137]:
# Base directory for the original images and the resized images
base_dir = "../ttv_split_ADNI_png_seq"
resized_base_dir = "../final_preprocessed2"

# Walk through all files under the 'data' directory
for root, _, files in walk(base_dir):
    # Check if the directory is a category directory (AD, MCI, CN)
    if "/AD/" in root:
        dtype = "AD"
    elif "/CN/" in root:
        dtype = "CN"
    elif "/MCI/" in root:
        dtype = "MCI"
    else:
        continue  # Skip any directories that don't belong to AD, MCI, CN

    # Check if it's a "train" or "test" directory
    if "test" in root:
        split = "test"
    elif "train" in root:
        split = "train"
    else:
        continue  # Skip if it's neither train nor test

    # Ensure the split category directory exists (train/test -> AD/CN/MCI)
    current_dir = path.join(resized_base_dir, split, dtype)
    if not path.isdir(current_dir):
        makedirs(current_dir, exist_ok=True)
        
    for file in files:
        if file.endswith(".png"):
            file_lst = file.split("_")
            subject_id = file_lst[0]  # Subject ID is the part before the first underscore
            dst_dir = path.join(resized_base_dir, split, dtype, subject_id)  # Path for the subject ID folder
            if not path.isdir(dst_dir):
                makedirs(dst_dir, exist_ok=True)
            center_crop_resize(path.join(root, file), path.join(dst_dir, file))

print("All images have been resized and saved!")

All images have been resized and saved!


In [144]:
import os
import shutil

# Define dataset root
dataset_root = "../final_preprocessed2"  # Update this path as needed
splits = ["train", "test"]  # Train and test directories
categories = ["AD", "MCI", "CN"]  # Categories

for split in splits:
    for category in categories:
        category_path = os.path.join(dataset_root, split, category)

        # Loop through each subject (ID) folder
        for subject_id in os.listdir(category_path):
            subject_path = os.path.join(category_path, subject_id)

            if not os.path.isdir(subject_path):  # Ensure it's a directory
                continue

            # Move all .nii files to the category folder
            for file in os.listdir(subject_path):
                if file.endswith(".png"):
                    src = os.path.join(subject_path, file)
                    dst = os.path.join(category_path, file)
                    shutil.move(src, dst)
                    # print(f"Moved {file} to {category_path}")

            # Remove the empty ID folder
            if not os.listdir(subject_path):  # Check if folder is empty
                os.rmdir(subject_path)
                print(f"Removed empty folder: {subject_path}")

print("Dataset restructuring complete!")

I87150_157.png
Moved I87150_157.png to ../final_preprocessed2/train/AD
I87150_143.png
Moved I87150_143.png to ../final_preprocessed2/train/AD
I87150_180.png
Moved I87150_180.png to ../final_preprocessed2/train/AD
I87150_142.png
Moved I87150_142.png to ../final_preprocessed2/train/AD
I87150_156.png
Moved I87150_156.png to ../final_preprocessed2/train/AD
I87150_168.png
Moved I87150_168.png to ../final_preprocessed2/train/AD
I87150_140.png
Moved I87150_140.png to ../final_preprocessed2/train/AD
I87150_154.png
Moved I87150_154.png to ../final_preprocessed2/train/AD
I87150_155.png
Moved I87150_155.png to ../final_preprocessed2/train/AD
I87150_141.png
Moved I87150_141.png to ../final_preprocessed2/train/AD
I87150_169.png
Moved I87150_169.png to ../final_preprocessed2/train/AD
I87150_145.png
Moved I87150_145.png to ../final_preprocessed2/train/AD
I87150_151.png
Moved I87150_151.png to ../final_preprocessed2/train/AD
I87150_179.png
Moved I87150_179.png to ../final_preprocessed2/train/AD
I87150